# Day 11 v2 — Gold: All Blob Marts (For Loop)

**Source:** Silver blob tables (Day 9) + Silver API dimension tables (Day 8)
**Sink:** `gold-volume/blob/` (4 Gold Delta marts)

| Gold Mart | Source tables | Grain | Key use case |
|---|---|---|---|
| `realtime_sessions_enriched` | realtime/charging_sessions + API dimensions | 1 row/session | IoT + business context |
| `invoice_summary` | invoices + API customers | 1 row/customer/month | Invoice analytics, billing |
| `kpi_sla_dashboard` | kpi_report + sla_report | 1 row/month | Executive KPI + SLA view |
| `state_performance` | state_breakdown + stations + weather + cities | 1 row/state/month | Geographic performance |

**All marts:** full overwrite in this v2 version.

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_REALTIME = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
SILVER_INVOICES = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices'
SILVER_REPORTS  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports'
SILVER_API      = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_BLOB       = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/blob'
PIPELINE        = 'pl_gold_blob_all_marts_v2'
RUN_TS          = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Gold blob : {GOLD_BLOB}')
print(f'RUN_TS    : {RUN_TS}')

In [ ]:
# Cell 3 — Read all Silver tables
rt_sessions  = spark.read.format('delta').load(f'{SILVER_REALTIME}/charging_sessions')
invoices     = spark.read.format('delta').load(SILVER_INVOICES)
kpi_report   = spark.read.format('delta').load(f'{SILVER_REPORTS}/kpi_report')
sla_report   = spark.read.format('delta').load(f'{SILVER_REPORTS}/sla_report')
state_bdown  = spark.read.format('delta').load(f'{SILVER_REPORTS}/state_breakdown')

# API dimensions
customers    = spark.read.format('delta').load(f'{SILVER_API}/customers')
stations     = spark.read.format('delta').load(f'{SILVER_API}/stations')
chargers     = spark.read.format('delta').load(f'{SILVER_API}/chargers')
tariffs      = spark.read.format('delta').load(f'{SILVER_API}/tariffs')
energy_prices= spark.read.format('delta').load(f'{SILVER_API}/energy_prices')
weather      = spark.read.format('delta').load(f'{SILVER_API}/weather')
cities       = spark.read.format('delta').load(f'{SILVER_API}/cities')
states_dim   = spark.read.format('delta').load(f'{SILVER_API}/states')

print('All Silver tables loaded')

In [ ]:
# Cell 4 — Build realtime_sessions_enriched
sta_dim = stations.select(
    'station_id',
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)
chg_dim = chargers.select(
    'charger_id', 'charger_type',
    F.col('power_kw').alias('charger_power_kw'),
    F.col('status').alias('charger_status'),
)
cust_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
)
tar_dim = tariffs.select(
    'tariff_id',
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
)
ep_win = Window.partitionBy('region').orderBy(F.col('effective_from').desc())
ep_dim = (
    energy_prices
    .withColumn('_rn', F.row_number().over(ep_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('region').alias('station_state'),
        F.col('price_per_kwh').alias('grid_price_per_kwh'),
    )
)

realtime_enriched = (
    rt_sessions
    .join(sta_dim,  on='station_id',   how='left')
    .join(chg_dim,  on='charger_id',   how='left')
    .join(cust_dim, on='customer_id',  how='left')
    .join(tar_dim,  on='tariff_id',    how='left')
    .join(ep_dim,   on='station_state', how='left')
    .withColumn('session_date',  F.to_date('plug_in_ts'))
    .withColumn('session_year',  F.year('plug_in_ts'))
    .withColumn('session_month', F.month('plug_in_ts'))
    .withColumn('session_hour',  F.hour('plug_in_ts'))
    .withColumn('estimated_cost_aud',
        F.when(F.col('tariff_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
             + F.col('tariff_price_per_min') * F.col('duration_min')).cast('decimal(10,2)')
        )
    )
    .withColumn('grid_cost_aud',
        F.when(F.col('grid_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('grid_price_per_kwh') * F.col('energy_kwh')).cast('decimal(10,2)')
        )
    )
    .withColumn('gross_margin_aud',
        F.when(F.col('estimated_cost_aud').isNotNull() & F.col('grid_cost_aud').isNotNull(),
            (F.col('estimated_cost_aud') - F.col('grid_cost_aud')).cast('decimal(10,2)')
        )
    )
    .withColumn('power_efficiency_pct',
        F.when(
            F.col('charger_power_kw').isNotNull() & (F.col('charger_power_kw') > 0)
            & F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('charger_power_kw') * F.col('duration_min') / 60) * 100
            ).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'realtime_sessions_enriched : {realtime_enriched.count()} rows')

In [ ]:
# Cell 5 — Build invoice_summary (per customer per month)
cust_id_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('email').alias('customer_email'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
)

# Extract customer_id from invoice_id pattern: INV-AU-<customer_suffix>-<seq>
# Bronze invoices store the path — extract customer context via join on invoice_number
# Since invoices don't have customer_id in Silver, we aggregate by month
invoice_summary = (
    invoices
    .groupBy('invoice_year', 'invoice_month')
    .agg(
        F.count('invoice_id').alias('total_invoices'),
        F.sum('file_size_kb').cast('decimal(12,2)').alias('total_file_size_kb'),
        F.avg('file_size_kb').cast('decimal(8,2)').alias('avg_file_size_kb'),
        F.min('invoice_date').alias('earliest_invoice_date'),
        F.max('invoice_date').alias('latest_invoice_date'),
        F.min('invoice_number').alias('invoice_number_min'),
        F.max('invoice_number').alias('invoice_number_max'),
    )
    .withColumn(
        'report_period',
        F.concat_ws('', F.col('invoice_year').cast('string'),
                    F.lpad(F.col('invoice_month').cast('string'), 2, '0'))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'invoice_summary : {invoice_summary.count()} rows')

In [ ]:
# Cell 6 — Build kpi_sla_dashboard (KPI + SLA merged on report_period)
kpi_sla_dashboard = (
    kpi_report.alias('kpi')
    .join(sla_report.alias('sla'), on='report_period', how='inner')
    .select(
        F.col('kpi.report_period'),
        F.col('kpi.report_year'),
        F.col('kpi.report_month'),
        F.col('kpi.generated_at').alias('kpi_generated_at'),
        # KPI metrics
        'kpi.total_sessions',
        'kpi.total_energy_kwh',
        'kpi.total_revenue_aud',
        'kpi.avg_session_duration_min',
        'kpi.avg_energy_per_session_kwh',
        # SLA metrics
        'sla.uptime_pct',
        'sla.avg_response_time_sec',
        'sla.incidents',
        'sla.mttr_hours',
        'sla.chargers_meeting_sla',
        'sla.total_chargers',
        # Derived
        F.when(F.col('sla.total_chargers') > 0,
            (F.col('sla.chargers_meeting_sla') / F.col('sla.total_chargers') * 100
            ).cast('decimal(6,2)')
        ).alias('sla_compliance_pct'),
        F.when(F.col('kpi.total_sessions') > 0,
            (F.col('kpi.total_revenue_aud') / F.col('kpi.total_sessions')).cast('decimal(8,2)')
        ).alias('revenue_per_session_aud'),
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'kpi_sla_dashboard : {kpi_sla_dashboard.count()} rows')

In [ ]:
# Cell 7 — Build state_performance (state breakdown + weather + station counts)

# Station count per state
station_counts = (
    stations.groupBy(F.col('state').alias('state_code'))
    .agg(
        F.count('station_id').alias('total_stations'),
        F.sum('total_chargers').alias('total_chargers'),
    )
)

# Latest weather per state (join cities → weather, aggregate to state level)
city_state = cities.select('city_id', F.col('state_code').alias('state_code'))
weather_state = (
    weather.join(city_state, on='city_id', how='left')
    .groupBy('state_code')
    .agg(
        F.avg('temperature_c').cast('decimal(5,2)').alias('avg_temp_c'),
        F.avg('humidity_pct').cast('decimal(5,2)').alias('avg_humidity_pct'),
        F.avg('wind_speed_kmh').cast('decimal(6,2)').alias('avg_wind_kmh'),
    )
)

# State name from states dimension
state_names = states_dim.select(
    F.col('state_code'),
    F.col('name').alias('state_name'),
    'country',
)

state_performance = (
    state_bdown
    .join(station_counts, on='state_code', how='left')
    .join(weather_state,  on='state_code', how='left')
    .join(state_names,    on='state_code', how='left')
    # Revenue per charger (normalised performance)
    .withColumn(
        'revenue_per_charger_aud',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('revenue_aud') / F.col('total_chargers')).cast('decimal(10,2)')
        )
    )
    .withColumn(
        'sessions_per_charger',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('sessions') / F.col('total_chargers')).cast('decimal(8,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'state_performance : {state_performance.count()} rows')

In [ ]:
# Cell 8 — Write all 4 Gold marts (full overwrite)
marts = [
    ('realtime_sessions_enriched', realtime_enriched,  ['session_year', 'session_month']),
    ('invoice_summary',            invoice_summary,    ['invoice_year', 'invoice_month']),
    ('kpi_sla_dashboard',          kpi_sla_dashboard,  ['report_year',  'report_month']),
    ('state_performance',          state_performance,  ['report_year',  'report_month']),
]

results = []
for mart_name, df, partition_cols in marts:
    path = f'{GOLD_BLOB}/{mart_name}'
    print(f'  Writing {mart_name} ...', end=' ')
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    rows = df.count()
    results.append({'mart': mart_name, 'rows': rows})
    print(f'OK ({rows} rows)')

print('\nAll Gold blob marts written.')

In [ ]:
# Cell 9 — Run summary + spot verify
print('=' * 60)
print('GOLD BLOB ALL MARTS v2 — RUN SUMMARY')
print('=' * 60)
print(f'  run_ts : {RUN_TS}')
print()
for r in results:
    print(f"  {r['mart']:<35} rows={r['rows']:>8}")

print('\nKPI + SLA Dashboard (all months):')
spark.read.format('delta').load(f'{GOLD_BLOB}/kpi_sla_dashboard') \
    .select('report_period', 'total_sessions', 'total_revenue_aud',
            'uptime_pct', 'sla_compliance_pct', 'revenue_per_session_aud') \
    .orderBy('report_period').show(truncate=False)

print('\nState performance (top 5 by revenue):')
spark.read.format('delta').load(f'{GOLD_BLOB}/state_performance') \
    .groupBy('state_code', 'state_name') \
    .agg(F.sum('revenue_aud').alias('total_revenue')) \
    .orderBy(F.col('total_revenue').desc()).show(5, truncate=False)